# app

Executable notebook module. English explanations precede each functional block. Restart the kernel after changing modules.


In [ ]:
from pathlib import Path
import json, sys, types
PROJECT = Path.cwd()
assert (PROJECT / 'app.ipynb').is_file(), 'Open this notebook from EcoSort-Demo.'

def load_notebook(name):
    if name in sys.modules:
        return sys.modules[name]
    path = PROJECT / (name + '.ipynb')
    notebook = json.loads(path.read_text())
    module = types.ModuleType(name)
    module.__file__ = str(path)
    sys.modules[name] = module
    try:
        for dependency in notebook['metadata'].get('dependencies', []):
            load_notebook(dependency)
        for cell in notebook['cells']:
            if cell['cell_type'] == 'code' and 'module' in cell.get('metadata', {}).get('tags', []):
                exec(compile(''.join(cell['source']), str(path), 'exec'), module.__dict__)
    except Exception:
        sys.modules.pop(name, None)
        raise
    return module

load_notebook('classifier')
load_notebook('disposal_rules')
__file__ = str(PROJECT / 'app.ipynb')


## Imports and configuration

Load required libraries and define the paths, labels and constants used by the following functions.


In [ ]:
"""English local interface for the project-trained MobileNetV3 classifier."""
import os
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"
import hashlib
import json
from pathlib import Path
import threading
import urllib.error
import urllib.request
import webbrowser

import gradio as gr
from PIL import Image
from classifier import TrainedPredictor
from disposal_rules import ITEM_TYPES, disposal_guidance

BASE = Path(__file__).resolve().parent
PORT = 7862
DISPLAY = {"battery":"Battery", "cardboard":"Cardboard", "ceramics":"Ceramics", "ewaste":"Electronic waste", "food_organics":"Food organics", "glass":"Glass", "metal":"Metal", "paper":"Paper", "plastic":"Plastic", "textile_trash":"Textiles", "trash":"Other waste", "vegetation":"Vegetation", "wood":"Wood", "mixed_items":"Mixed / unspecified material", "household_problem_waste":"Household problem waste", "light_bulbs":"Light bulbs", "medicines":"Medicines"}




## EcoSort

Group related state and operations for this module.

- `__init__`: Initialize stored settings and model state.
- `classify`: Validate inputs, predict material and apply human-confirmed disposal guidance.


In [ ]:
class EcoSort:
    def __init__(self):
        self.classifier = TrainedPredictor()  # Load the trained checkpoint once.
        self.lock = threading.Lock()  # Prevent simultaneous requests from calling the shared model.

    def classify(self, image, item_type="Not confirmed", food_service=False):
        if not isinstance(image, Image.Image):  # Reject missing or invalid image inputs.
            raise gr.Error("Upload an image or capture a photo first.")
        item_type = item_type or "Not confirmed"
        if item_type not in ITEM_TYPES:
            raise gr.Error("Unknown item confirmation.")
        with self.lock:
            result = self.classifier.identify(image, food_service)
        route, guidance = result["route"], result["guidance"]
        if item_type != "Not confirmed":
            route, guidance = disposal_guidance(result["label"], item_type, food_service)  # Change disposal guidance only; do not change model scores or weights.
        status = "Unable to identify reliably. Top candidate" if result["uncertain"] else "Model suggestion"
        decision = f"{status}: {result['object']} | {DISPLAY[result['label']]} | {route}"
        return {DISPLAY[k]:v for k,v in result["scores"].items()}, decision, guidance


## verify_release

Check the model file against its recorded identity and training metadata.


In [ ]:
def verify_release():
    """Check that the saved model matches its training result."""
    results = json.loads((BASE / "model/results.json").read_text())
    settings = json.loads((BASE / "model/settings.json").read_text())
    training = results["training"]
    assert training["status"] == "completed"
    assert settings["training_images"] == training["split_counts"]["train"]
    assert training["verified_image_count"] == sum(training["split_counts"].values())
    assert hashlib.sha256((BASE / "model/model.keras").read_bytes()).hexdigest() == results["model_sha256"]


## launch

Build the English upload/webcam interface and start the local server.


In [ ]:
def launch(prevent_thread_lock=False):
    verify_release()
    url = f"http://127.0.0.1:{PORT}"
    try:
        with urllib.request.urlopen(url+"/config", timeout=1) as response:
            config = json.load(response)
        if config.get("title") != "EcoSort AI - Demo":
            raise RuntimeError(f"Port {PORT} belongs to another application.")
        webbrowser.open(url)
        return None
    except urllib.error.URLError:
        pass
    engine = EcoSort()
    # shortcut: pinned Gradio frontend exports enforce English; recheck after upgrading.
    core = next((Path(gr.__file__).parent/"templates/frontend/assets").glob("core-*.js")).name
    english = f"<script type='module'>import {{setupi18n, changeLocale}} from '/assets/{core}'; await setupi18n({{}}, 'en'); changeLocale('en'); document.documentElement.lang = 'en';</script>"  # Force built-in upload and camera controls to use English.
    with gr.Blocks(title="EcoSort AI - Demo", analytics_enabled=False) as demo:
        gr.Markdown("# EcoSort AI")
        with gr.Row():
            with gr.Column():
                image = gr.Image(sources=["upload", "webcam"], type="pil", label="Waste item", height=450)  # Allow upload or webcam input and pass a PIL image to Python.
                item_type = gr.Dropdown(ITEM_TYPES, value="Not confirmed", label="Confirmed item type")
                food_service = gr.Checkbox(value=False, label="I have access to a City of Sydney food scraps service")
                with gr.Row():
                    submit = gr.Button("Identify item", variant="primary")
                    clear = gr.ClearButton([image, item_type], value="Clear")
            with gr.Column():
                decision = gr.Textbox(label="Identification and disposal route", lines=3, interactive=False)
                scores = gr.Label(num_top_classes=5, label="Relative candidate scores (not calibrated probabilities)")  # Show five leading scores without promising real-world probabilities.
                guidance = gr.Markdown()
        clear.add([food_service, scores, decision, guidance])
        submit.click(engine.classify, [image, item_type, food_service], [scores, decision, guidance])  # Connect button inputs to classify and route its three return values.
        image.change(lambda: ("Not confirmed", {}, "", ""), outputs=[item_type, scores, decision, guidance])  # A different image invalidates old confirmation and predictions.
        item_type.change(lambda: ("", ""), outputs=[decision, guidance])
        food_service.change(lambda: ("", ""), outputs=[decision, guidance])
    demo.launch(share=False, server_name="127.0.0.1", server_port=PORT, inbrowser=True,  # Serve locally without a public sharing tunnel.
                prevent_thread_lock=prevent_thread_lock, show_error=True, footer_links=[], head=english)
    return demo
